## A notebook to figure out how to convert matrix to long format

- we need a mapper - or something 'which columns should be considered'. Can be multiple
- we need some way to come up with a type (what is measured) and a unit (in what units) - if information is unavailable
- we need a way to enter those types and units which ARE known.
- we need a way to kind of assess what is already known or there so we can figure out what to do next
  if it's not immediately clear it;s timeseries data, we need to come up wit ha tracking number or repeated measured number or whatnot

Basically, matrix-to-long is the harder problem. Probably. Since long-format is cumbersome, but also a bit simpler since everything you need to know is already there.


In [1]:
import pandas as pd
from tqdm import tqdm
import numpy as np

In [2]:
infile = 'Melb_data/GaitRite_HISSPhase1_GRW_spatiotemporal_all_testdata.csv'
df=pd.read_csv(infile)

In [3]:
df.columns

Index(['SubjectID', 'Session', 'Func_test', 'Pt_id', 'TestSetId', 'id', 'Time',
       'Distance', 'Amb_Time', 'Velocity', 'Step_Count', 'Cadence',
       'Step_Time_L', 'Step_Time_R', 'Step_Len_L', 'Step_Len_R',
       'Cycle_Time_L', 'Cycle_Time_R', 'Stride_Len_L', 'Stride_Len_R',
       'Supp_Base_L', 'Supp_Base_R', 'Swing_Perc_L', 'Swing_Perc_R',
       'Swing_Time_L', 'Swing_Time_R', 'Stance_Perc_L', 'Stance_Perc_R',
       'Stance_Time_L', 'Stance_Time_R', 'S_supp_Perc_L', 'S_supp_Perc_R',
       'S_Supp_Time_L', 'S_Supp_Time_R', 'D_Supp_PercL', 'D_Supp_PercR',
       'D_Supp_TimeL', 'D_Supp_TimeR', 'ToeInOutL', 'ToeInOutR',
       'HeelOffOnTimeL', 'HeelOffOnTimeR', 'HeelOffOnPercL', 'HeelOffOnPercR',
       'D_SuppLoadTm_L', 'D_SuppLoadTm_R', 'D_SuppLoadPerc_L',
       'D_SuppLoadPerc_R', 'D_SuppUnloadTm_L', 'D_SuppUnloadTm_R',
       'D_SuppUnloadPerc_L', 'D_SuppUnloadPerc_R', 'StrideVelocity_L',
       'StrideVelocity_R', 'StepLen_SD_L', 'StepLen_SD_R', 'StepTm_SD_L',
       

In [4]:
df_cols = pd.read_csv('Melb_data/Information_columns.csv')

In [5]:
# checking to see what is there:
convert_slice = slice(6, -1)
var_types = df_cols.iloc[convert_slice]['Column'].tolist()
var_units = df_cols.iloc[convert_slice]['Unit'].tolist()
var_units = [x if not pd.isna(x) else 'NotDefined' for x in var_units]

In [6]:
new_df = pd.DataFrame(columns=['participant_id','trial','repeat', 'variable', 'value', 'unit'])
print(new_df)

Empty DataFrame
Columns: [participant_id, trial, repeat, variable, value, unit]
Index: []


In [7]:
df[var_types]

,Time,Distance,Amb_Time,Velocity,Step_Count,Cadence,Step_Time_L,Step_Time_R,Step_Len_L,Step_Len_R,...,HeelOffOn_SD_R,SuppBase_SD_L,SuppBase_SD_R,Ft_Length_L,Ft_Length_R,Ft_Width_L,Ft_Width_R,Step_Time_Dif,Step_Len_Dif,Cycle_Time_Dif
0,45069.65625,478.720001,2.37,134.600006,3,112.699997,1.080,1.088,65.363998,66.889000,...,0.011,2.218,0.000,26.799999,29.900000,8.4,8.9,0.026,0.801,0.009
1,45069.65625,245.649994,5.84,165.100006,7,129.099998,1.054,0.538,71.357002,87.000999,...,0.006,0.327,1.098,31.000000,27.299999,9.8,8.9,0.008,0.063,0.000
2,45069.65625,289.769989,5.38,173.300003,7,89.000000,0.988,0.011,75.440002,107.492996,...,0.006,2.975,2.645,29.600000,27.299999,10.6,10.6,0.008,2.026,0.001
3,45069.65694,347.839996,2.82,155.199997,3,151.599998,1.008,0.013,64.875000,66.940002,...,0.012,0.803,1.679,22.700001,23.500000,7.0,6.4,0.005,2.161,0.021
4,45069.65694,242.779999,3.63,148.399994,3,92.599998,1.036,0.019,46.695000,79.740002,...,0.000,0.000,1.188,31.200001,28.900000,11.0,11.4,0.008,2.915,0.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6322,45174.50833,263.899994,5.15,125.400002,3,66.099998,0.546,0.583,87.092003,73.900002,...,0.000,0.000,6.192,28.400000,25.299999,9.4,10.0,0.017,1.849,0.013
6323,45174.50833,414.880005,2.74,102.000000,5,118.199997,0.092,0.064,85.968002,72.320999,...,0.000,0.000,5.484,29.400000,28.500000,9.3,8.3,0.026,0.369,0.009
6324,45174.50833,166.809998,2.92,133.000000,4,85.400002,0.084,0.575,76.200996,111.461998,...,0.000,0.000,0.016,25.400000,28.200001,7.4,7.8,0.007,0.205,0.004
6325,45174.50833,242.880005,2.88,149.900002,3,94.500000,0.508,0.003,52.107002,69.382004,...,0.022,4.974,0.830,22.700001,23.200001,8.1,7.9,0.013,0.982,0.008


In [8]:
convert_stuff = {'participant_id': 'SubjectID', 'trial': 'Session'}

In [9]:
# allright - now let's go through all of the entries in the nmatrix:


pt_and_sess = [convert_stuff['participant_id'], convert_stuff['trial']]
prev_pt, prev_sess = [None, None]
repeat=0

unique_pts = set(df[convert_stuff['participant_id']].tolist())
unique_sess = set(df[convert_stuff['trial']].tolist())


n_pt_sess = df[[convert_stuff['participant_id'], convert_stuff['trial']]].drop_duplicates().shape[0]
pbar = tqdm(total=n_pt_sess*len(var_types))
data_list = []

for i, p in enumerate(unique_pts):
    for j, s in enumerate(unique_sess):
        # i, row in df[pt_and_sess + var_types].iterrows():
        subset = df[(df[convert_stuff['participant_id']] == p) & (df[convert_stuff['trial']] == s)]
        
        for var_type, var_unit in zip(var_types, var_units):

            vals = subset[var_type].tolist()
            

            # new_stuff = {'participant_id': [p for x in range(len(vals))],
            #             'trial':  [s for x in range(len(vals))],
            #              'repeat': [x+1 for x in range(len(vals))],
            #              'variable': [var_type for x in range(len(vals))],
            #              'value': vals, 
            #              'unit': [var_unit for x in range(len(vals))]
            #             }

            for k, val in enumerate(vals):
                data_list.append([p, s, k+1, var_type, val, var_unit])
                
            pbar.update(1)
            
            # data_list.append(new_stuff)
            

my_new_df_cols = ['participant_id', 'trial', 'repeat', 'variable', 'value', 'unit']


 94%|███████████████████████████████  | 36191/38500 [00:02<00:00, 19937.10it/s]

In [10]:
my_new_df = pd.DataFrame(data_list, columns=my_new_df_cols)

In [11]:
my_new_df

,participant_id,trial,repeat,variable,value,unit
0,4624,S5_25H,1,Time,45077.45972,ms
1,4624,S5_25H,2,Time,45077.45972,ms
2,4624,S5_25H,3,Time,45077.45972,ms
3,4624,S5_25H,4,Time,45077.45972,ms
4,4624,S5_25H,5,Time,45077.45972,ms
...,...,...,...,...,...,...
487174,4606,S2_1H,8,Cycle_Time_Dif,0.00400,NotDefined
487175,4606,S2_1H,9,Cycle_Time_Dif,0.02900,NotDefined
487176,4606,S2_1H,10,Cycle_Time_Dif,0.00400,NotDefined
487177,4606,S2_1H,11,Cycle_Time_Dif,0.02100,NotDefined


In [12]:
from pathlib import Path

In [13]:
# Default: put generated file in synthesized/
outdir = Path.cwd() / "temp"
outdir.mkdir(parents=True, exist_ok=True)

outfile = outdir / f"{Path(infile).stem}_long.csv"

In [14]:
outfile

PosixPath('/data1/projects/Heat/temp/GaitRite_HISSPhase1_GRW_spatiotemporal_all_testdata_long.csv')

In [15]:
my_new_df.to_csv(outfile, index=False)